# 27. From Logits to Text

From *Transformers from Scratch*.

In [ ]:
# On Colab there is nothing on disk, so this fetches the two or
# three files the chapter uses. If you cloned the repository it
# finds them where they already are and downloads nothing.
import os
import sys
import urllib.request

BASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/code/"
PLACES = [".", "code", "../code"]

for name in ["ch27_decoding.py"]:
    if not any(os.path.exists(os.path.join(where, name))
               for where in PLACES):
        urllib.request.urlretrieve(BASE + name, name)

for where in PLACES:
    if os.path.isdir(where) and where not in sys.path:
        sys.path.insert(0, where)

Turning a row of scores into a word, five ways.

The model produces one score per token. Every strategy in this file is a different answer to the same question: which one do we take?

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True, floatmode="fixed")

WORDS = ["milk", "water", "tea", "coffee", "juice", "beer", "soup", "ink"]
LOGITS = np.array([3.2, 2.6, 1.9, 1.5, 0.9, 0.4, -0.6, -2.0])

In [ ]:
def softmax(scores, temperature=1.0):
    scaled = np.asarray(scores, dtype=float) / temperature
    shifted = scaled - scaled.max()
    e = np.exp(shifted)
    return e / e.sum()

In [ ]:
def greedy(logits):
    return int(np.argmax(logits))

Keep the k largest, renormalise, and discard everything else.

In [ ]:
def top_k_filter(probs, k):
    kept = np.zeros_like(probs)
    order = np.argsort(probs)[::-1][:k]
    kept[order] = probs[order]
    return kept / kept.sum()

Keep the smallest group whose total reaches p.

In [ ]:
def nucleus_filter(probs, p):
    order = np.argsort(probs)[::-1]
    running = np.cumsum(probs[order])
    cut = int(np.searchsorted(running, p)) + 1
    kept = np.zeros_like(probs)
    kept[order[:cut]] = probs[order[:cut]]
    return kept / kept.sum()

Push down the score of anything already produced.

In [ ]:
def repetition_penalty(logits, used, strength=1.2):
    out = np.array(logits, dtype=float)
    for token in used:
        out[token] = (out[token] / strength if out[token] > 0
                      else out[token] * strength)
    return out

In [ ]:
def main():
    probs = softmax(LOGITS)
    print("ONE ROW OF SCORES, AND THE PROBABILITIES THEY BECOME")
    for word, score, p in zip(WORDS, LOGITS, probs):
        print(f"  {word:<8} score {score:>5.1f}   probability {p:.4f}")

    print("\nGREEDY TAKES THE LARGEST, EVERY TIME")
    print(f"  {WORDS[greedy(LOGITS)]}, with probability "
          f"{probs.max():.4f}")
    print("  the same input always gives the same output")

    print("\nTEMPERATURE CHANGES HOW DECISIVE THE CHOICE IS")
    print("  T      top word   its share   words above 1 per cent")
    for t in [0.25, 0.7, 1.0, 1.5, 3.0]:
        scaled = softmax(LOGITS, t)
        print(f"  {t:<6} {WORDS[int(scaled.argmax())]:<10} "
              f"{scaled.max():<11.4f} {int((scaled > 0.01).sum())}")

    print("\nTOP-K KEEPS A FIXED NUMBER")
    for k in [1, 3, 5]:
        kept = top_k_filter(probs, k)
        alive = [w for w, v in zip(WORDS, kept) if v > 0]
        print(f"  k={k}  {', '.join(alive)}")

    print("\nNUCLEUS KEEPS AS MANY AS IT TAKES TO REACH A SHARE")
    for p in [0.5, 0.8, 0.95]:
        kept = nucleus_filter(probs, p)
        alive = [w for w, v in zip(WORDS, kept) if v > 0]
        print(f"  p={p}  {len(alive)} words: {', '.join(alive)}")
    print("  a flat distribution keeps many, a sharp one keeps few,")
    print("  which is the whole reason it is preferred to top-k")

    print("\nWHAT A REPETITION PENALTY DOES")
    used = [0, 1]
    penalised = softmax(repetition_penalty(LOGITS, used))
    print("  word      before   after")
    for i, word in enumerate(WORDS[:4]):
        print(f"  {word:<9} {probs[i]:.4f}   {penalised[i]:.4f}")

    print("\nSAMPLING THE SAME ROW TWENTY TIMES")
    rng = np.random.default_rng(0)
    for name, filtered in [("plain", probs),
                           ("top-k 3", top_k_filter(probs, 3)),
                           ("nucleus 0.9", nucleus_filter(probs, 0.9))]:
        draws = rng.choice(len(WORDS), size=20, p=filtered)
        counts = {WORDS[i]: int((draws == i).sum()) for i in set(draws)}
        shown = ", ".join(f"{w} {c}" for w, c in sorted(counts.items()))
        print(f"  {name:<13} {shown}")

In [ ]:
main()

The chapter this came from works through every number above, and explains why each step is there.